# Kafka Consumer

## Initialize Kafka Cluster & Databricks Secrets

In [0]:
# API_KEY = dbutils.secrets.get(scope = "grandmasters", key = "API_KEY")
# API_SECRET = dbutils.secrets.get(scope = "grandmasters", key = "API_SECRET")
# BOOTSTRAP_SERVERS = "pkc-l7pr2.ap-south-1.aws.confluent.cloud:9092"

BOOTSTRAP_SERVERS = "pkc-921jm.us-east-2.aws.confluent.cloud:9092"
API_KEY           = "Z2DLXTSRMR3JXMAX"
API_SECRET        = "cfltMorayXn5QEmOFSKYwuh4gdbDuyTRZa4cQZQ775OEqpYkKw1b61I5MOUN3V3w"

## Consumer Configuration

In [0]:
KAFKA_TOPIC = "lichess.tv.event"
BRONZE_TABLE = "grandmasters.bronze.tv_feed"
CHECKPOINT_LOCATION = "/Volumes/grandmasters/bronze/checkpoints/tv_feed"
TRIGGER_INTERVAL = "5 seconds"
sasl_config = (
    f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="{API_KEY}" password="{API_SECRET}";'
)

spark.sql("""
          CREATE VOLUME IF NOT EXISTS grandmasters.bronze.checkpoints
        COMMENT 'Checkpoint storage for streaming jobs';
          """)

## Stream Processing

### Step 1 - Initialize Kafka Stream DF
Check if Connection to Kafka Stream is successful

In [0]:
from pyspark.sql.functions import col, get_json_object

kafka_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers",  BOOTSTRAP_SERVERS)
    .option("subscribe",                KAFKA_TOPIC)
    # "earliest" guarantees processing from the very beginning of the topic retention window
    .option("startingOffsets",          "earliest") 
    
    # Consumer & Provider Handshake Security
    .option("kafka.security.protocol",  "SASL_SSL")
    .option("kafka.sasl.mechanism",     "PLAIN")
    .option("kafka.sasl.jaas.config",   sasl_config)
    
    # Admin Client Handshake (Resolves connection dropping on initialization)
    .option("kafka.admin.security.protocol", "SASL_SSL")
    .option("kafka.admin.sasl.mechanism",    "PLAIN")
    .option("kafka.admin.sasl.jaas.config",  sasl_config)
    
    # Backpressure: Protects cluster from out-of-memory errors on massive fresh backlogs
    .option("maxOffsetsPerTrigger",     "5000") 
    .load()
)

# Uncomment to check if the stream is working
#display(kafka_stream_df)

### Step 2 - STRIP & PARSE LAYER (Optimized & Typed)
Cast binary payload once to conserve driver and executor CPU cycles

In [0]:
string_payload_df = kafka_stream_df.select(
    col("value").cast("string").alias("payload_str")
)

# Uncomment to debug
#display(string_payload_df)

### Step 3 - Map to Bronze Stream DF columns

In [0]:
bronze_stream_df = string_payload_df.select(
    col("payload_str").alias("raw_event"),
    get_json_object(col("payload_str"), "$.t").alias("event_type"),
    get_json_object(col("payload_str"), "$.game_id").alias("game_id"),
    get_json_object(col("payload_str"), "$.move_seq").cast("int").alias("move_seq"),
    get_json_object(col("payload_str"), "$.ingested_at").cast("timestamp").alias("ingested_at")
)

### Step 4 - Query streams and Write to table

In [0]:
query = (
    bronze_stream_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .option("mergeSchema", "false") 
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable(BRONZE_TABLE)
)

print(f"Stream started. Writing to {BRONZE_TABLE} every {TRIGGER_INTERVAL}.")
print("Run the cells below in a separate notebook or after stopping this stream.\n")
print(f"Stream ID  : {query.id}")

query.awaitTermination()